# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [3]:
# import pypyodbc
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.colors import to_hex

# 2.0 Import spot position and size QA data

In [4]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)

df.head(2)



,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

In [6]:
df.value_counts("MachineName"), df.value_counts("Device"), df.value_counts("Energy")

(MachineName
 Gantry 3    10576
 Gantry 1    10472
 Gantry 4    10357
 Gantry 2     9767
 Name: count, dtype: int64,
 Device
 XRV-3000    31815
 XRV-4000     9357
 Name: count, dtype: int64,
 Energy
 150    8250
 240    8243
 200    8233
 100    8232
 70     8214
 Name: count, dtype: int64)

# 4.0 filtering data

## calculate abs shift

In [7]:
sub_df = df[["ADate", "MachineName", "Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

## calculate expected spot positions
pred_xrv4000 = {
    'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175],
    'Top-Left': [-125, -125], 'Top-Centre': [0, -125], 'Top-Right': [125, -125],
    'Left': [-125, 0], 'Centre': [0, 0], 'Right': [125, 0],
    'Bottom-Left': [-125, 125], 'Bottom-Centre': [0, 125], 'Bottom-Right': [125, 125],
    'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]
}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

# first calculate absolute shift
sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

# convert absolute shift to relative shift wrt Centre spot
group_cols = ["ADate", "MachineName", "Device", "Energy", "Gantry Angle"]

centre_shift = (
    sub_df[sub_df["Spot"] == "Centre"]
    .groupby(group_cols, as_index=False)[["abs_xpos", "abs_ypos"]]
    .mean()
    .rename(columns={"abs_xpos": "centre_abs_x", "abs_ypos": "centre_abs_y"})
)

sub_df = sub_df.merge(centre_shift, on=group_cols, how="left")

# create relative shift columns
sub_df["rel_xpos"] = sub_df["abs_xpos"] - sub_df["centre_abs_x"]
sub_df["rel_ypos"] = sub_df["abs_ypos"] - sub_df["centre_abs_y"]

In [8]:
print(sub_df.head(2))

                ADate MachineName  Energy    Device  Gantry Angle  \
0 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   
1 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   

            Spot     x-pos     y-pos  px_pos  py_pos  abs_xpos  abs_ypos  \
0  Bottom-Centre   -0.2357  124.9224       0     125   -0.2357   -0.0776   
1    Bottom-Left -125.0153  125.4501    -125     125   -0.0153    0.4501   

   centre_abs_x  centre_abs_y  rel_xpos  rel_ypos  
0       -0.5678       -0.0179    0.3321   -0.0597  
1       -0.5678       -0.0179    0.5525    0.4680  


# Helper for HLS palette

In [9]:
def get_hls_palette_hex(n_colors):
    """Return HLS palette as hex colors."""
    return [to_hex(c) for c in sns.color_palette("hls", n_colors=n_colors)]

# Main plotting functions

In [10]:
def plotly_spot_position(df, pos, gantry, device, energy, gantry_angle, n_months, exclude_centre=False):
    """Plot spot position time series data."""
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    selected_df = selected_df.sort_values("ADate")

    palette_hex = get_hls_palette_hex(selected_df["Spot"].nunique())

    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot',
        color='Spot',
        color_discrete_sequence=palette_hex,
        title=f'{gantry} - relative shift wrt Centre - {pos}',
        labels={'ADate': 'Date', pos: 'Relative shift (mm)'},
        height=500
    )

    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    fig.update_traces(
        mode='markers+lines',
        marker=dict(size=12, line=dict(width=2), opacity=0.65),
        line=dict(width=1)
    )

    fig.show()

In [11]:
# plotting relative x andy-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last 24 months
plotly_spot_position(sub_df, "rel_xpos", "Gantry 4", "XRV-3000", 70, 0, 24, exclude_centre=True)
plotly_spot_position(sub_df, "rel_ypos", "Gantry 4", "XRV-4000", 70, 0, 24, exclude_centre=True)

## plot another device

In [12]:
plotly_spot_position(sub_df, "rel_xpos", "Gantry 2", "XRV-3000", 70, 0, 24, exclude_centre=True)
plotly_spot_position(sub_df, "rel_ypos", "Gantry 2", "XRV-4000", 70, 0, 24, exclude_centre=True)

In [13]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)

selected_df = sub_df[
    (sub_df["MachineName"] == "Gantry 2") &
    (sub_df["Device"] == "XRV-3000") &
    (sub_df['ADate'] >= start_date)
].copy()

# Calculate average relative x-pos per adate and energy
selected_df['avg_rel_xpos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["rel_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,centre_abs_x,centre_abs_y,rel_xpos,rel_ypos,avg_rel_xpos
32499,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Bottom-Centre,0.3273,125.5616,0,125,0.3273,0.5616,0.158,0.4836,0.1693,0.0780,0.0922
32500,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Bottom-Left,-124.7349,125.5062,-125,125,0.2651,0.5062,0.158,0.4836,0.1071,0.0226,0.0922
32501,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Bottom-Right,125.6069,125.4004,125,125,0.6069,0.4004,0.158,0.4836,0.4489,-0.0832,0.0922
32502,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Centre,0.1580,0.4836,0,0,0.1580,0.4836,0.158,0.4836,0.0000,0.0000,0.0922
32503,2025-04-24 18:57:29,Gantry 2,70,XRV-3000,0,Left,-124.9901,0.9085,-125,0,0.0099,0.9085,0.158,0.4836,-0.1481,0.4249,0.0922


In [14]:
def plotly_ave_spot_position(df, parameter, gantry, device, n_months, exclude_centre=True):
    """Plot average relative spot position across all spot positions with the same adate and energy."""
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    avg_df = (
        selected_df
        .groupby(['ADate', 'Energy', 'Gantry Angle'], as_index=False)[parameter]
        .mean()
        .rename(columns={parameter: 'avg_rel_pos'})
        .sort_values("ADate")
    )

    avg_df['Energy'] = avg_df['Energy'].astype(int).astype(str)

    fig = px.scatter(
        avg_df,
        x='ADate',
        y='avg_rel_pos',
        symbol='Gantry Angle',
        color='Energy',
        title=f'average relative shift wrt Centre: {parameter}',
        labels={'ADate': 'Date', 'avg_rel_pos': 'Average relative shift (mm)'},
        height=500
    )

    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    fig.update_traces(
        mode='markers+lines',
        marker=dict(size=12, line=dict(width=2), opacity=0.65),
        line=dict(width=1)
    )

    fig.show()

In [15]:
plotly_ave_spot_position(sub_df, "rel_xpos", "Gantry 1", "XRV-3000", 24, exclude_centre=True)

# Attempt 0A — same shape, different color (using G2, 70 MeV, GA0)

In [16]:
def plotly_spot_position_color_only(df, pos, gantry, device, energy, gantry_angle, n_months, exclude_centre=True):
    """Plot relative shift using same marker shape for all spots, different colours only."""
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    selected_df = selected_df.sort_values("ADate")
    palette_hex = get_hls_palette_hex(selected_df["Spot"].nunique())

    fig = px.scatter(
        selected_df,
        x="ADate",
        y=pos,
        color="Spot",
        color_discrete_sequence=palette_hex,
        title=f"{gantry} - relative shift wrt Centre - colour only - {pos}",
        labels={"ADate": "Date", pos: "Relative shift (mm)"},
        height=500
    )

    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    fig.update_traces(
        mode="markers+lines",
        marker=dict(size=11, symbol="circle", line=dict(width=1), opacity=0.65),
        line=dict(width=1)
    )

    fig.show()

In [17]:
plotly_spot_position_color_only(sub_df, "rel_xpos", "Gantry 2", "XRV-3000", 70, 0, 24)

Observation:
Colour helps separate the spots, but with many spots on one plot the graph still feels crowded.
Similar colours can be hard to distinguish, especially where points overlap.

# Attempt 0B — same color, different shape

In [18]:
# Attempt 0B
# Question:
# Is marker shape alone enough to distinguish spots clearly?

def plotly_spot_position_shape_only(df, pos, gantry, device, energy, gantry_angle, n_months, exclude_centre=True):
    """Plot relative shift using same colour for all spots, different marker shapes only."""

    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    fig = px.scatter(
        selected_df,
        x="ADate",
        y=pos,
        symbol="Spot",
        title=f"{gantry} - relative shift wrt Centre - shape only - {pos}",
        labels={"ADate": "Date", pos: "Relative shift (mm)"},
        height=500
    )

    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    fig.update_traces(
        mode="markers+lines",
        marker=dict(size=11, color="royalblue", line=dict(width=1)),
        line=dict(width=1, color="royalblue")
    )

    fig.show()

In [19]:
plotly_spot_position_shape_only(sub_df, "rel_xpos", "Gantry 2", "XRV-3000", 70, 0, 24)

Observation:
Shape gives some separation, but with many spot categories the legend becomes harder to scan.
Overlapping markers and lines are still difficult to follow when colour is not also used.

# Attempt 0C — different color and different shape

In [20]:
# Attempt 0C
# Question:
# Is using both colour and shape the clearest way to distinguish spots?

def plotly_spot_position_color_and_shape(df, pos, gantry, device, energy, gantry_angle, n_months, exclude_centre=True):
    """Plot relative shift using both colour and shape to distinguish spots."""

    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    palette = sns.color_palette("deep", n_colors=selected_df["Spot"].nunique())
    palette_hex = [to_hex(c) for c in palette]

    fig = px.scatter(
        selected_df,
        x="ADate",
        y=pos,
        color="Spot",
        symbol="Spot",
        color_discrete_sequence=palette_hex,
        title=f"{gantry} - relative shift wrt Centre - colour + shape - {pos}",
        labels={"ADate": "Date", pos: "Relative shift (mm)"},
        height=500
    )

    fig.add_hline(y=1, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    fig.update_traces(
        mode="markers+lines",
        marker=dict(size=11, line=dict(width=1)),
        line=dict(width=1)
    )

    fig.show()

In [21]:
plotly_spot_position_color_and_shape(sub_df, "rel_xpos", "Gantry 2", "XRV-3000", 70, 0, 24)

Observation:
Using both colour and shape makes the plot easier to read than using either one alone.
Spot identities are easier to distinguish in both the graph and the legend.
This is still somewhat crowded, but it is the clearest version among the three single-panel attempts.

# Attempt 1 — same plot, but facet by gantry angle

In [22]:
def plotly_spot_position_facet_angle(df, pos, gantry, device, energy, n_months, exclude_centre=True):
    """Plot relative shift time series, faceted by Gantry Angle."""

    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date) &
        (df["Energy"] == energy)
    ].copy()

    if exclude_centre:
        selected_df = selected_df[selected_df["Spot"] != "Centre"]

    fig = px.scatter(
        selected_df,
        x="ADate",
        y=pos,
        color="Spot",
        symbol="Spot",
        facet_col="Gantry Angle",
        facet_col_wrap=2,
        title=f"{gantry} - {device} - {energy} MeV - {pos} faceted by Gantry Angle",
        labels={"ADate": "Date", pos: "Relative shift (mm)"},
        height=800
    )

    fig.add_hline(y=1, line_dash="dash", line_color="grey")
    fig.add_hline(y=-1, line_dash="dash", line_color="grey")

    fig.update_traces(
        mode="markers+lines",
        marker=dict(size=9, line=dict(width=1)),
        line=dict(width=1)
    )

    fig.show()

In [23]:
# Attempt 1
# Question:
# Does relative shift behave differently at different gantry angles?
plotly_spot_position_facet_angle(sub_df, "rel_xpos", "Gantry 2", "XRV-3000", 70, 24)

# Attempt 2 — facet by spot instead of colouring by spot

In [24]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plotly_spot_position_grid_spot(
    df,
    pos,
    gantry,
    device,
    energy,
    gantry_angle,
    n_months,
    exclude_centre=True
):
    """Plot relative shift time series in physical spot layout, with blank Centre slot."""

    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["ADate"] >= start_date) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle)
    ].copy()

    selected_df = selected_df.sort_values("ADate")

    # Physical layout
    spot_grid = [
        ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right"],
        ["Top-Left", "Top-Centre", "Top-Right"],
        ["Left", "Centre", "Right"],
        ["Bottom-Left", "Bottom-Centre", "Bottom-Right"],
        ["Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]
    ]

    flat_spots = [s for row in spot_grid for s in row]

    # HLS colours
    palette_hex = [to_hex(c) for c in sns.color_palette("hls", n_colors=len(flat_spots))]
    color_map = dict(zip(flat_spots, palette_hex))

    fig = make_subplots(
        rows=5,
        cols=3,
        subplot_titles=flat_spots,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.05,
        horizontal_spacing=0.05
    )

    for r, row_spots in enumerate(spot_grid, start=1):
        for c, spot in enumerate(row_spots, start=1):

            # leave Centre blank if excluding it
            if spot == "Centre" and exclude_centre:
                fig.add_hline(y=1, line_dash="dash", line_color="grey", row=r, col=c)
                fig.add_hline(y=-1, line_dash="dash", line_color="grey", row=r, col=c)
                continue

            spot_df = selected_df[selected_df["Spot"] == spot].copy()

            if not spot_df.empty:
                fig.add_trace(
                    go.Scatter(
                        x=spot_df["ADate"],
                        y=spot_df[pos],
                        mode="markers+lines",
                        name=spot,
                        showlegend=False,
                        marker=dict(
                            size=7,
                            color=color_map[spot],
                            opacity=0.65,
                            line=dict(width=1, color=color_map[spot])
                        ),
                        line=dict(width=1, color=color_map[spot])
                    ),
                    row=r,
                    col=c
                )

            fig.add_hline(y=1, line_dash="dash", line_color="grey", row=r, col=c)
            fig.add_hline(y=-1, line_dash="dash", line_color="grey", row=r, col=c)

    fig.update_yaxes(range=[-1, 1])

    for r in range(1, 6):
        fig.update_yaxes(title_text="Relative shift (mm)", row=r, col=1)

    for c in range(1, 4):
        fig.update_xaxes(title_text="Date", row=5, col=c)

    fig.update_layout(
        title=f"{gantry} - {device} - {energy} MeV - angle {gantry_angle} - {pos} by Spot",
        height=1300,
        width=1000
    )

    fig.show()

In [25]:
# Attempt 2
# Question:
# Is one specific spot drifting more than others?
plotly_spot_position_grid_spot(
    sub_df,
    "rel_ypos",
    "Gantry 4",
    "XRV-4000",
    70,
    0,
    24,
    exclude_centre=True
)